In [1]:
import json
import re
from collections import Counter

import ollama
import pdfplumber

In [2]:
def clean_labeled_output(labeled_data, verbose=True):
    """Remove empty entries and normalize whitespace (strip \\n, \\t, extra spaces)."""
    cleaned = []
    for item in labeled_data:
        text = item.get("text", "")
        normalized = re.sub(r"\s+", " ", text).strip()

        if normalized:
            cleaned.append({"text": normalized, "label": item.get("label")})
        elif verbose:
            print(
                f"Dropped empty entry with label: {item.get('label')}"
            )  # just to check what it drops

    return cleaned

## Text extraction from PDF

In [3]:
# 1. Determine the document's own "body text" baseline (font + size)
def get_body_font_profile(pdf):
    """
    Scan all characters in the PDF and determine the dominant font name and
    size, i.e. what "normal paragraph text" looks like in this document.
    Headers/labels/emphasis are then detected as deviations from this
    baseline, rather than by matching a specific font-naming pattern.
    """
    font_sizes = Counter()
    font_names = Counter()

    for page in pdf.pages:
        for char in page.chars:
            font_sizes[round(char["size"], 1)] += 1
            font_names[char["fontname"]] += 1

    if not font_sizes or not font_names:
        # No character-level data available (e.g. scanned/image PDF) -
        # caller should fall back to plain extract_text() in this case.
        return None, None

    body_size = font_sizes.most_common(1)[0][0]
    body_font = font_names.most_common(1)[0][0]
    return body_size, body_font


# 2. Classify individual characters as "emphasized" relative to that baseline
def is_emphasized(char, body_size, body_font):
    """
    True if a character looks visually emphasized relative to the document's
    own body text - either a different font family (bold/weight variants
    usually show up here) or a meaningfully larger size.
    """
    different_font = char["fontname"] != body_font
    is_italic_variant = any(
        tag in char["fontname"] for tag in ("Italic", "Oblique", "italic", "oblique")
    )
    looks_bold = different_font and not is_italic_variant
    looks_bigger = char["size"] > body_size + 0.5
    return looks_bold or looks_bigger


def is_italic(char, body_font):
    return any(
        tag in char["fontname"] for tag in ("Italic", "Oblique", "italic", "oblique")
    )


# 3. Group words into lines, wrap emphasized/italic runs with markers
def _word_is_emphasized(word_chars, body_size, body_font):
    """A word counts as emphasized if a majority of its characters are."""
    if not word_chars:
        return False
    flags = [is_emphasized(c, body_size, body_font) for c in word_chars]
    return sum(flags) > len(flags) / 2


def _word_is_italic(word_chars, body_font):
    if not word_chars:
        return False
    flags = [is_italic(c, body_font) for c in word_chars]
    return sum(flags) > len(flags) / 2


def _extract_page_lines(page, body_size, body_font):
    """
    Rebuild each line of the page from character-level data, wrapping
    emphasized runs in **...** and italic runs in _..._ so the LLM can use
    them as classification signals downstream.
    """
    chars = page.chars
    if not chars:
        return []

    # Group chars into words using pdfplumber's own word extraction for
    # correct spacing/boundaries, then map each word back to its chars
    # (by matching x0/top) to evaluate emphasis per word.
    words = page.extract_words(extra_attrs=["fontname", "size"])

    # Build a lookup of chars by rounded (top) position for fast grouping
    # into lines; pdfplumber words already carry "top" for this purpose.
    lines_map = {}
    for w in words:
        top_key = round(w["top"], 1)
        lines_map.setdefault(top_key, []).append(w)

    # For emphasis, re-derive per-word char list by spatial overlap.
    def chars_for_word(w):
        return [
            c
            for c in chars
            if c["top"] >= w["top"] - 1
            and c["bottom"] <= w["bottom"] + 1
            and c["x0"] >= w["x0"] - 0.5
            and c["x1"] <= w["x1"] + 0.5
        ]

    output_lines = []
    for top_key in sorted(lines_map.keys()):
        line_words = sorted(lines_map[top_key], key=lambda w: w["x0"])
        rendered = []
        open_bold, open_italic = False, False

        for w in line_words:
            wchars = chars_for_word(w)
            emph = _word_is_emphasized(wchars, body_size, body_font)
            ital = _word_is_italic(wchars, body_font)

            # close markers that no longer apply
            if open_italic and not ital:
                rendered.append("_")
                open_italic = False
            if open_bold and not emph:
                rendered.append("**")
                open_bold = False

            # open markers that newly apply
            if emph and not open_bold:
                rendered.append("**")
                open_bold = True
            if ital and not open_italic:
                rendered.append("_")
                open_italic = True

            rendered.append(w["text"])

        if open_italic:
            rendered.append("_")
        if open_bold:
            rendered.append("**")

        line_text = " ".join(rendered)
        # clean up marker artifacts from consecutive open/close with no content between
        line_text = line_text.replace("** **", " ").replace("_ _", " ")
        output_lines.append(line_text)

    return output_lines


# 4. Top-level function: PDF path -> full_text string ready for the LLM
def extract_text_for_llm(pdf_path, join_pages_with="\n\n"):
    """
    Full pipeline: given a PDF path, return a single text string with
    **bold** and _italic_ markers preserved, suitable for passing to
    classify_entire_document(full_text).

    Falls back to plain text extraction (no formatting markers) for any
    page where character-level font data isn't available (e.g. a scanned
    or image-only page), so the pipeline never silently drops content.
    """
    pages_text = []

    with pdfplumber.open(pdf_path) as pdf:
        body_size, body_font = get_body_font_profile(pdf)

        for page in pdf.pages:
            if body_size is None:
                # No usable font metadata anywhere in the doc - plain fallback
                text = page.extract_text()
                if text:
                    pages_text.append(text)
                continue

            lines = _extract_page_lines(page, body_size, body_font)
            if lines:
                pages_text.append("\n".join(lines))
            else:
                # This page had no extractable words (e.g. scanned image) -
                # fall back to plain extraction so we don't lose the page.
                text = page.extract_text()
                if text:
                    pages_text.append(text)

    return join_pages_with.join(pages_text)

In [4]:
SAMPLES = range(1, 11)
full_texts = {}
for n in SAMPLES:
    full_texts[n] = extract_text_for_llm(f"../data/input/pdfs/sample{n}.pdf")
    print(f"sample{n}: {len(full_texts[n])} chars")

sample1: 6282 chars


sample2: 14909 chars


sample3: 7089 chars


sample4: 6640 chars


sample5: 7133 chars
sample6: 1777 chars
sample7: 1740 chars


sample8: 5468 chars


sample9: 7750 chars


sample10: 5962 chars


## Section labeling

### Prompt

In [5]:
SYSTEM_PROMPT = """
You are a highly accurate data extraction assistant. Your task is to classify lines and paragraphs of text from a Data Management Plan into one of the following five categories:

1. "title": the single main title of the document (appears once, typically at the top, usually short).
2. "section.title": a heading that opens a new top-level section. Often starts with a letter prefix (A., B., C.) or a named phrase like "Element 1:".
3. "section.description": Explanatory or instructional text about what a section covers, NOT phrased as a direct question to the researcher, and NOT the researcher's own response. Typically appears right after a section.title and before any question.text or answer.text.
4. "question.text": A specific question, instruction, or prompt that asks the researcher to address a particular topic. Usually ends in a colon or is phrased as a direct ask.
5. "answer.text": The researcher's actual written response — narrative text describing what the team will do, has done, or plans to do, usually in first- or third-person about the research team.

FORMATTING MARKERS:
Text wrapped in **double asterisks** was visually emphasized (e.g. bold or larger) in the source PDF. Text wrapped in _underscores_ was italicized. Use these as supporting evidence, not strict rules:
- A short emphasized phrase (often ending in a period or colon) at the start of a line, followed by longer plain text, often functions as a label, heading, or question that is embedded in the same paragraph as its answer in the source layout. In that case, split the emphasized phrase from the text that follows it and classify them separately, even though they appeared on the same line.
- Longer emphasized or italicized passages that read as instructional or explanatory (describing what a section should contain, rather than asking the researcher something directly or presenting the researcher's response) are usually section.description.
- Not all emphasis indicates a label or heading — some documents use bold/italics for ordinary emphasis within an answer. Use the surrounding context and what the text actually says to decide, not the formatting alone.
- Absence of formatting markers does not mean text can't be a title, section.title, or question.text — some documents don't use bold/italic for structure at all. Rely on lexical and positional context in that case, as before.

RULES:
- Process the entire document. Classify every heading, question, description, and paragraph — do not skip or summarize any of it.
- Reproduce each "text" value verbatim from the source, EXCLUDING the ** and _ formatting markers themselves — strip them out before writing the "text" field.
- If a section has no distinct answer (e.g., a heading is immediately followed by only descriptive or list-style content with no separate researcher response), classify what's there once — do not invent an empty or placeholder entry to fill an "answer.text" slot.
- Some sections may lack a question.text entirely if the section.title is immediately followed by descriptive content or an answer — do not force text into "question.text" if it doesn't read as a direct question or prompt

EXAMPLE:
title: “DATA MANAGEMENT AND SHARING PLAN” or “Center for Bio-Inspired Energy Science” 
Section.title: “Element 1: Data Type” or “1. Data sharing and preservation “
Section.description: “Data management plans should describe whether and how data generated in the course of the proposed research will be shared and preserved. If the plan is not to share and/or preserve certain data, then the plan must explain the basis of the decision (for example, cost/benefit considerations, other parameters of feasibility, scientific appropriateness, or limitations discussed in #4). At a minimum, DMPs must describe how data sharing and preservation will enable validation of results, or how results could be validated if data are not shared or preserved. “ 
Question.text: “A. Types and amount of scientific data expected to be generated in the project” or “Data Types and Sources. A brief, high-level description of the data to be generated or used through the course of the proposed research and which of these are considered Digital Research Data necessary to validate the research findings.”
Answer.text: “This secondary data analysis project will analyze deidentified data from 48,218 participants from eight studies and the publicly available NHANES cohorts (wrist NHANES 2011-2014; hip/counts-NHANES 2003-2006). The studies include (i) the RISE Study, (ii) the SOL-VIDA Study, (iii) the iWATCH Study, (iv) the MOCA Study, (v) the PHASE Study, (vi) the AusDiab Study, (vii) the ACT Study, and (viii) the WHISH accelerometer substudy.” or “ For the proposed research, Director Samuel Stupp with help from the Executive Director of Research will take the lead and responsibility for coordinating and ensuring data storage and access and communicating expectations to all investigators. However, all senior investigators will also be involved in managing, storing, and disseminating the results of the project and will be responsible for checking that the plan is being followed.
"""

### LLM Prompting

In [6]:
schema = {
    "type": "array",
    "items": {
        "type": "object",
        "properties": {
            "text": {"type": "string"},
            "label": {
                "type": "string",
                "enum": [
                    "title",
                    "section.title",
                    "section.description",
                    "question.text",
                    "answer.text",
                ],
            },
        },
        "required": ["text", "label"],
    },
}

def classify_entire_document(full_text):
    model = "qwen2.5:14b"
    print(f"Sending full text to {model}")

    response = ollama.chat(
        model=model,
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {
                "role": "user",
                "content": f"Classify the following text into the required JSON format:\n\n{full_text}",
            },
        ],
        format=schema, # enforce output to follow schema
        options={"num_ctx": 8192, "num_predict": -1},
    )

    result = json.loads(response["message"]["content"])
    return clean_labeled_output(result)

In [7]:
results = {}
for n in SAMPLES:
    print(f"=== sample{n} ===")
    results[n] = classify_entire_document(full_texts[n])

=== sample1 ===
Sending full text to qwen2.5:14b


=== sample2 ===
Sending full text to qwen2.5:14b


=== sample3 ===
Sending full text to qwen2.5:14b


=== sample4 ===
Sending full text to qwen2.5:14b


=== sample5 ===
Sending full text to qwen2.5:14b


=== sample6 ===
Sending full text to qwen2.5:14b


=== sample7 ===
Sending full text to qwen2.5:14b


=== sample8 ===
Sending full text to qwen2.5:14b


=== sample9 ===
Sending full text to qwen2.5:14b


=== sample10 ===
Sending full text to qwen2.5:14b


### Clean up result

In [8]:
for n in SAMPLES:
    print()
    print('=' * 70)
    print(f'sample{n} -- {len(results[n])} items')
    print('=' * 70)
    for item in results[n]:
        print(f"  [{item['label']:<20}] {item['text'][:90]}")


sample1 -- 28 items
  [title               ] DATA MANAGEMENT AND SHARING PLAN
  [section.title       ] Element 1: Data Type:
  [question.text       ] A. Types and amount of scientific data expected to be generated in the project:
  [answer.text         ] This secondary data analysis project will analyze deidentified data from 48,218 participan
  [question.text       ] B. Scientific data that will be preserved and shared, and the rationale for doing so:
  [answer.text         ] As this is a secondary data analysis project, we will only be able to publicly share in th
  [question.text       ] C. Metadata, other relevant data, and associated documentation:
  [answer.text         ] In addition to the data described above, code and models will be included in the repositor
  [section.title       ] Element 2: Related Tools, Software and/or Code:
  [answer.text         ] Data will be analyzed with custom code by our statistical and computer science team. ActiG
  [section.title       ] Element

### Save results

One JSON file per sample, easy to open and review individually.


In [9]:
from pathlib import Path

OUT_DIR = Path('visual-signals-results')
OUT_DIR.mkdir(parents=True, exist_ok=True)

for n in SAMPLES:
    out_path = OUT_DIR / f'sample{n}.json'
    out_path.write_text(
        json.dumps(results[n], indent=2, ensure_ascii=False), encoding='utf-8'
    )

print(f'saved {len(SAMPLES)} files to {OUT_DIR.resolve()}/')
for n in SAMPLES:
    print(f'  sample{n}.json  ({len(results[n])} items)')

saved 10 files to C:\Users\Nahid\dmpbridge\notebooks\visual-signals-results/
  sample1.json  (28 items)
  sample2.json  (30 items)
  sample3.json  (14 items)
  sample4.json  (26 items)
  sample5.json  (33 items)
  sample6.json  (11 items)
  sample7.json  (9 items)
  sample8.json  (17 items)
  sample9.json  (14 items)
  sample10.json  (17 items)
